# ADLS assessment appeals to Silver and AI

Reads appeal Parquet through the `BronzeLakehouse` shortcut, writes a conformed Silver fact, and uses Fabric AI functions to create governed text signals. The original narrative is preserved.

In [ ]:
%%configure -f
{
  "defaultLakehouse": { "name": "SilverLakehouse" }
}

In [ ]:
bronze_lakehouse_item = "BronzeLakehouse"
appeal_file_path = "Files/appeals/assessment_appeals.parquet"

In [ ]:
import re
import requests
from pyspark.sql import functions as F

import notebookutils
workspace_id = notebookutils.runtime.context["currentWorkspaceId"]

def resolve_item_id(display_name, item_type):
    token = notebookutils.credentials.getToken("pbi")
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items?type={item_type}"
    response = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=60)
    response.raise_for_status()
    matches = [item for item in response.json()["value"] if item["displayName"] == display_name]
    if len(matches) != 1:
        raise ValueError(f"Expected one {item_type} named '{display_name}', found {len(matches)}")
    return matches[0]["id"]

def snake_case(name):
    return re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name).lower()

bronze_id = resolve_item_id(bronze_lakehouse_item, "Lakehouse")
source_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{bronze_id}/{appeal_file_path}"
raw = spark.read.parquet(source_path)
for column in raw.columns:
    raw = raw.withColumnRenamed(column, snake_case(column))

In [ ]:
appeals = (raw
    .withColumn("tax_year", F.col("tax_year").cast("int"))
    .withColumn("submitted_date", F.to_date("submitted_date"))
    .withColumn("resolved_date", F.to_date("resolved_date"))
    .withColumn("requested_adjustment_pct", F.col("requested_adjustment_pct").cast("double"))
    .withColumn("synthetic", F.col("synthetic").cast("boolean"))
    .dropna(subset=["appeal_id", "parcel_id", "reason_code", "narrative"])
    .dropDuplicates(["appeal_id"]))

assert appeals.filter(F.col("synthetic") != True).count() == 0
assert appeals.filter(F.length(F.trim("narrative")) == 0).count() == 0
(appeals.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("dbo.fact_appeal"))
print(f"Conformed appeals: {appeals.count()}")

## AI enrichment

The following cell requires Fabric AI functions to be enabled. It adds decision-support signals but does not approve, reject, or rank appeals.

In [ ]:
import pandas as pd
import synapse.ml.aifunc as aifunc

pdf = appeals.toPandas()
text = pdf["narrative"].astype(str)
pdf["ai_sentiment"] = text.ai.analyze_sentiment()
pdf["ai_summary"] = text.ai.summarize()
pdf["ai_reason_code"] = text.ai.classify(
    "COMPARABLE_SALES", "PROPERTY_CONDITION", "DATA_CORRECTION",
    "CLASSIFICATION", "RENOVATION_TIMING", "INFORMATION_REQUEST"
)
pdf["ai_follow_up"] = text.ai.classify("urgent_follow_up", "standard_review")
pdf["enriched_at_utc"] = pd.Timestamp.utcnow().isoformat()

enriched = spark.createDataFrame(pdf)
(enriched.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("dbo.fact_appeal_ai"))

In [ ]:
evaluation = (spark.table("dbo.fact_appeal_ai")
    .withColumn("enrichment_success", F.col("ai_sentiment").isNotNull() & F.col("ai_summary").isNotNull() & F.col("ai_reason_code").isNotNull())
    .withColumn("sentiment_match", F.coalesce(F.lower("ai_sentiment") == F.lower("ground_truth_sentiment"), F.lit(False)))
    .withColumn("reason_match", F.coalesce(F.upper("ai_reason_code") == F.upper("reason_code"), F.lit(False))))

display(evaluation.groupBy("ground_truth_sentiment", "ai_sentiment").count().orderBy("ground_truth_sentiment", "ai_sentiment"))
display(evaluation.agg(
    F.count("*").alias("records"),
    F.avg(F.col("sentiment_match").cast("double")).alias("sentiment_accuracy"),
    F.avg(F.col("reason_match").cast("double")).alias("reason_code_accuracy"),
    (F.lit(1.0) - F.avg(F.col("enrichment_success").cast("double"))).alias("enrichment_failure_rate"),
))